In [ ]:
%%file view.py
import canvas_helpers as H
from ipywidgets import Output
from ipycanvas import MultiCanvas, hold_canvas
from IPython.display import display


class View:
    def __init__(self, game, size=240, margin=20):
        self.game = game
        self.size = size
        self.margin = margin
        self.dx = (size-2*margin)/game.size
        self.boardspec = (margin, margin, self.dx, self.dx, self.game.size, self.game.size)

        self.out = Output(layout={'border': '1px solid black'})
        self.mcanvas = MultiCanvas(4, width=size, height=size, layout={'border': '1px solid black'})
        self.bg, self.mg, self.info, self.fg = self.mcanvas
        self.mg.fill_style = '#CCCCCC'
        H.draw_grid(self.fg, self.boardspec, line_width=2, color='black')

        self.game.observe(self.update)
        self.game.new_game()

    def new_game(self, **kwargs):
        self.mcanvas.clear()
        self.mg.fill_rect(self.margin, self.margin, self.size-2*self.margin)
        H.draw_grid(self.fg, self.boardspec, line_width=2, color='black')

        for pos in self.game.mines:
            H.place_stone(self.bg, pos, self.boardspec, radius=0.2, color='black')

        self.out.append_stdout(f'New Game. Find the {self.game.n_mines} mines.\n')

    def reveal(self, **kwargs):
        with hold_canvas():
            for pos in kwargs['reveal']:
                H.clear_field(self.mg, pos, self.boardspec)

    def flag(self, **kwargs):
        pos = kwargs['pos']
        if kwargs['status']:
            H.place_flag(self.info, pos, self.boardspec, color='red')
        else:
            H.clear_field(self.info, pos, self.boardspec)

    def win(self, **kwargs):
        self.out.append_stdout(f'Congrats, you revealed all mines!\n')
        for c, r in view.game.mines:
            if not self.game.mines_grid[r][c]:
                H.place_flag(self.info, (c, r), self.boardspec, color='red')

    def game_over(self, **kwargs):
        self.out.append_stdout(f'BOOM! Game over!\n')
        self.info.clear()
        self.mg.clear()

    def update(self, event,  **kwargs):
        self.out.append_stdout(f'{event}, {kwargs}\n')
        if hasattr(self, event):
            getattr(self, event)(**kwargs)

    def _ipython_display_(self):
        display(self.mcanvas, self.out)

In [ ]:
from minesweeper import Game

In [ ]:
game = Game()
view = View(game)
view

In [ ]:
view.mg.clear()

In [ ]:
game.new_game()
game.mines

In [ ]:
game.mines

In [ ]:
game.toggle_flag(1,1)

In [ ]:
view.game_over()

In [ ]:
 self._notify('game_over', mines_gird=self.mines_grid)
            redraw_board(canvas)
            output_area.clear_output()
            show_game_over_message()

_notify('win')
            redraw_board(canvas)
            show_win_message()
              self.flag_all_mines()
            return

self._notify('reveal')
        redraw_board(canvas) 
    reveal_all_mines()

self._notify('new_game, size=.self.size, n_mines=self.n_mines)
        # Log-Bereich einmal löschen bei neuem Spiel
        output_area.clear_output()
        log('Neues Spiel gestartet.')
        log(f'Spielfeld: {GRID_SIZE}x{GRID_SIZE}, Minen: {NUMBER_OF_MINES}')
        log(f'Flaggen-Modus: {'aktiv' if flag_mode_button.value else 'inaktiv'}')
             redraw_board(canvas)    

self._notify('flag', status=self.flag_grid[row][col])
        status = 'gesetzt' if flag_grid[row][col] else 'entfernt'
        log(f'Flagge {status} auf Feld ({row}, {col}).')
        redraw_board(canvas)

In [ ]:
def redraw_board(canvas):
    '''
    Zeichnet das komplette Spielfeld neu über das Darstellungsmodul.
    '''
    D.update(
        canvas,
        BOARD_SPEC,
        event='redraw',
        mines_grid=mines_grid,
        visibility_grid=visibility_grid,
        flag_grid=flag_grid,
        neighbor_mine_counts=neighbor_mine_counts,
    )

In [ ]:
def draw_board(canvas, board_spec,
               mines_grid, visibility_grid,
               flag_grid, neighbor_mine_counts):
    '''
    Zeichnet das komplette Spielfeld neu.

    Args:
        canvas: ipycanvas.Canvas.
        board_spec (tuple): (x0, y0, dx, dy, ncol, nrow).
        mines_grid (list[list[bool]]): Minenpositionen.
        visibility_grid (list[list[bool]]): Sichtbarkeitsstatus.
        flag_grid (list[list[bool]]): gesetzte Flaggen.
        neighbor_mine_counts (list[list[int]]): Nachbarminenanzahl.
    '''
    canvas.clear()

    nrow = len(mines_grid)
    ncol = len(mines_grid[0])

    for row in range(nrow):
        for col in range(ncol):
            pos = (col, row)

            if visibility_grid[row][col]:
                # Aufgedecktes Feld: weiss
                H.fill_field(canvas, pos, board_spec, color='#FFFFFF')

                if mines_grid[row][col]:
                    # Mine zeichnen
                    draw_mine(canvas, board_spec, pos)
                else:
                    count = neighbor_mine_counts[row][col]
                    if count > 0:
                        draw_number(canvas, board_spec, pos, count)

            else:
                # Verdecktes Feld: grau
                H.fill_field(canvas, pos, board_spec, color='#CCCCCC')
                if flag_grid[row][col]:
                    draw_flag(canvas, board_spec, pos)

    # Gitterlinien oben drauf
    H.draw_grid(canvas, board_spec, line_width=1, color='black')


def update(canvas, board_spec, event, **kwargs):
    '''
    Zentrale Update-Funktion für Darstellungsereignisse.

    Unterstützte Events:
      - 'new_game'
      - 'redraw'

    Args:
        canvas: ipycanvas.Canvas.
        board_spec (tuple): (x0, y0, dx, dy, ncol, nrow).
        event (str): Bezeichner des Ereignisses.
        **kwargs: weitere Daten (z.B. Grids) je nach Event.
    '''
    if event in ('new_game', 'redraw'):
        draw_board(
            canvas,
            board_spec,
            kwargs['mines_grid'],
            kwargs['visibility_grid'],
            kwargs['flag_grid'],
            kwargs['neighbor_mine_counts'],
        )

In [ ]:
import random
import traceback


from ipycanvas import Canvas
from ipywidgets import VBox, HBox, Button, ToggleButton, Output
from IPython.display import display

import darstellung as D

# --------------------------------------------------
# Konfiguration des Spiels
# --------------------------------------------------

GRID_SIZE = 10       # Anzahl Zeilen und Spalten des Spielfelds
NUMBER_OF_MINES = 10    # Anzahl Minen im Spielfeld

CELL_SIZE = 20          # Pixelfläche eines Feldes (Breite und Höhe)

BOARD_SPEC = (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE, GRID_SIZE, GRID_SIZE)


output_area = Output()

# Toggle-Button für Flaggenmodus
flag_mode_button = ToggleButton(
    description='Flaggen-Modus',
    value=False,
    tooltip='Aktiv: Klick = Flagge setzen/entfernen\nInaktiv: Klick = Feld aufdecken',
)

In [ ]:
@output_area.capture(clear_output=False)
def handle_mouse_click(x, y, canvas):
    '''
    Reagiert auf Mausklicks im Spielfeldbereich.

    Steuerung:
    - Linksklick (Flaggen-Modus AUS)  -> Feld aufdecken
    - Linksklick (Flaggen-Modus EIN)  -> Flagge setzen/entfernen

    Args:
        x (float): x-Koordinate im Canvas.
        y (float): y-Koordinate im Canvas.
    '''
    x0, y0, dx, dy, ncol, nrow = BOARD_SPEC
    log(f'Klick bei Canvas-Koordinate ({x:.1f}, {y:.1f})')
    log(f'Flaggen-Modus aktuell: {'aktiv' if flag_mode_button.value else 'inaktiv'}')

    # Prüfen, ob der Klick innerhalb des Spielfeldbereichs liegt
    if not (x0 <= x < x0 + ncol * dx and y0 <= y < y0 + nrow * dy):
        log('Klick ausserhalb des Spielfeldes – ignoriert.')
        return

    col = int((x - x0) // dx)
    row = int((y - y0) // dy)
    log(f'Berechnetes Feld: row={row}, col={col}')

    try:
        if flag_mode_button.value:
            toggle_flag(row, col, canvas)
        else:
            reveal_cell(row, col, canvas)
    except Exception:
        print('FEHLER IM CLICK-HANDLER:')
        traceback.print_exc()


@output_area.capture(clear_output=False)
def on_flag_mode_change(change):
    '''
    Callback, wenn der Flaggen-Modus-Button umgeschaltet wird.

    Args:
        change (dict): Änderungsinformation von ipywidgets.
    '''
    if change['name'] == 'value':
        status = 'aktiv' if change['new'] else 'inaktiv'
        log(f'Flaggen-Modus umgeschaltet: {status}')


@output_area.capture(clear_output=False)
def new_game_clicked(button, canvas):
    '''
    Event-Handler für den 'Neues Spiel'-Button.

    Args:
        button: Das Button-Objekt, das das Event ausgelöst hat.
    '''
    log("Button 'Neues Spiel' geklickt.")
    initialize_game(canvas)


# --------------------------------------------------
# Startfunktion
# --------------------------------------------------

def start_game(grid_size=10, nmines=10):
    '''
    Startet das Minesweeper-Spiel im Jupyter-Notebook.

    - initialisiert das Spielfeld
    - verbindet Maus-Events und Button-Events mit dem Canvas bzw. den Widgets
    - zeigt Canvas, Buttons und das Log-Output-Widget an
    '''
    global GRID_SIZE, NUMBER_OF_MINES, BOARD_SPEC
    GRID_SIZE = grid_size
    NUMBER_OF_MINES = nmines
    BOARD_SPEC = (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE, GRID_SIZE, GRID_SIZE)

    CANVAS_SIZE = (GRID_SIZE + 2) * CELL_SIZE

    canvas_config = {
        'width': CANVAS_SIZE,
        'height': CANVAS_SIZE,
        'layout': {
            'border': '1px solid black',
            'width': f'{CANVAS_SIZE}px',
            'height': f'{CANVAS_SIZE}px',
            'min_width': f'{CANVAS_SIZE}px',
            'min_height': f'{CANVAS_SIZE}px',
            'max_width': f'{CANVAS_SIZE}px',
            'max_height': f'{CANVAS_SIZE}px',
        },
    }

    canvas = Canvas(**canvas_config) 



    initialize_game(canvas)

    # Maus-Events
    canvas.on_mouse_down(lambda x, y: handle_mouse_click(x, y, canvas))

    # Flaggen-Modus-Änderungen beobachten
    flag_mode_button.observe(on_flag_mode_change, names='value')

    new_game_button = Button(description='Neues Spiel')
    new_game_button.on_click(lambda bt: new_game_clicked(bt, canvas))

    buttons_row = HBox([new_game_button, flag_mode_button])

    # ui = VBox([canvas, buttons_row, output_area])
    # display(ui)
    display(canvas, buttons_row, output_area)